# Stage 1A: FNO + Adam, three seeds

Reference baseline for every later comparison. Same split, data, epochs and learning rate as the main
notebook; only the seed changes.

| setting | value |
|---|---|
| model | FNO (neuralop), 2,410,177 params |
| optimizer | Adam, lr $10^{-3}$ |
| loss | MSE (standardized units) |
| epochs / batch | 20 / 32 |
| seeds | 0, 1, 2 |

## Setup

In [ ]:
import sys, glob, os, importlib.util
if importlib.util.find_spec("neuralop") is None:
    !pip install -q neuraloperator
# project code (nsbench, optim, solver) comes from the private ns-surrogate-code dataset
code_dir = next((os.path.dirname(os.path.dirname(p)) for p in
                 glob.glob("/kaggle/input/**/nsbench/__init__.py", recursive=True)), os.path.abspath(".."))
sys.path.insert(0, code_dir)
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nsbench
from nsbench import Trainer, save_csv, save_fig, save_predictions, run_record
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.DataFrame([{"python": sys.version.split()[0], "torch": torch.__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "code": code_dir}])

## Data

In [ ]:
D = nsbench.load()
pd.DataFrame([{"data": D.data_dir, "train batches": len(D.train_loader), "val batches": len(D.val_loader),
               "test samples": len(D.x_test), "y_mean": D.y_mean.item(), "y_std": D.y_std.item()}])

## Training

In [ ]:
SEEDS, EPOCHS, LR = [0, 1, 2], 20, 1e-3
rows, hists = [], []
for s in SEEDS:
    tr = Trainer(nsbench.FourierNeuralOperator, lambda p: torch.optim.Adam(p, lr=LR),
                 D.train_loader, D.val_loader, D.test_loader, D.y_mean, D.y_std, device,
                 seed=s, label=f"FNO Adam seed {s}").fit(EPOCHS)
    rows.append(run_record(tr.summary(), run="FNO", optimizer="Adam", lr=LR, seed=s,
                           epochs=EPOCHS, batch=32, loss_fn="MSE", monitor="loss"))
    hists.append(tr.history().assign(seed=s))
    save_predictions(tr, D.x_test, f"fno_adam_seed{s}")
runs = pd.concat(rows, ignore_index=True)
save_csv(runs, "stage1_fno_adam_seeds")
save_csv(pd.concat(hists, ignore_index=True), "stage1_fno_adam_history")
runs[["label", "best_epoch", "test_rel_L2", "test_mse_norm", "opt_state_MB", "peak_gpu_MB", "sec_per_epoch"]]

## Mean ± std over seeds

In [ ]:
cols = ["test_rel_L2", "test_mse_norm", "best_val_loss", "opt_state_MB", "peak_gpu_MB", "sec_per_epoch"]
agg = runs[cols].agg(["mean", "std"]).T
agg["n_seeds"] = len(runs)
save_csv(agg, "stage1_fno_adam_mean_std", index=True)
agg

## Validation curves

In [ ]:
H = pd.concat(hists, ignore_index=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for s, h in H.groupby("seed"):
    ax[0].plot(h.epoch, h.val_mse_norm, label=f"seed {s}")
    ax[1].plot(h.epoch, h.val_rel_L2, label=f"seed {s}")
ax[0].set(xlabel="epoch", ylabel="validation MSE (standardized)", yscale="log")
ax[1].set(xlabel="epoch", ylabel="validation relative $L^2$", yscale="log")
ax[0].legend(); fig.suptitle("FNO + Adam, three seeds"); fig.tight_layout()
save_fig("stage1_fno_adam_curves", fig)